# Installtion 

In [ ]:
!pip install numpy pandas matplotlib seaborn scikit-learn xgboost \
sentence-transformers transformers torch scikit-multilearn joblib

#  Imports 

In [ ]:

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")

# Core
import numpy as np
import pandas as pd
import torch

# Visualization
import matplotlib.pyplot as plt

# ML Preprocessing
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Classical ML Models
from sklearn.multioutput import MultiOutputClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Metrics
from sklearn.metrics import (
    hamming_loss,
    f1_score,
    jaccard_score,
)

# Multi-label splitting
from skmultilearn.model_selection import iterative_train_test_split

# Sentence Embeddings
from sentence_transformers import SentenceTransformer

# Transformers (HuggingFace)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Model saving
import joblib
import json




# Load data


In [ ]:
df = pd.read_csv("data\clean_wiki_movies.csv")

In [42]:

# 3. Handle text length - truncate very long plots
MAX_PLOT_LENGTH = 5000
df['Plot_Clean'] = df['Plot_Clean'].str[:MAX_PLOT_LENGTH]


#  CREATE MULTILABEL TARGETS


In [43]:
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df['Genre_List'])

print(f"Total unique genres: {len(mlb.classes_)}")
print(f"Target matrix shape: {y.shape}")
print(f"\nGenre classes (first 20): {mlb.classes_[:20]}")

Total unique genres: 18
Target matrix shape: (25023, 18)

Genre classes (first 20): ['action' 'animation' 'biography' 'black_comedy' 'children' 'comedy'
 'crime' 'documentary' 'drama' 'fantasy' 'history' 'musical' 'mystery'
 'romance' 'series' 'short' 'supernatural' 'thriller']


#  TRAIN-TEST SPLIT


In [44]:

X = df['Plot_Clean'].values

# Convert to format needed for iterative stratification
X_indices = np.arange(len(X)).reshape(-1, 1)

# Perform iterative stratified split
X_train_idx, y_train, X_test_idx, y_test = iterative_train_test_split(
    X_indices, y, test_size=0.2
)

# Get actual data
X_train = X[X_train_idx.flatten()]
X_test = X[X_test_idx.flatten()]

In [45]:
X_train.shape, X_test.shape


((20015,), (5008,))

# APPROACH : CLASSICAL ML WITH TF-IDF


##  Create TF-IDF features


In [ ]:

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    strip_accents='unicode',
    lowercase=True,
    stop_words='english'
)

In [ ]:

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


## MODEL 1: MULTINOMIAL NAIVE BAYES


In [ ]:

nb_model = MultiOutputClassifier(MultinomialNB())
nb_model.fit(X_train_tfidf, y_train)

MultiOutputClassifier(estimator=MultinomialNB())

In [ ]:
y_pred_nb = nb_model.predict(X_test_tfidf)


In [ ]:
hamming_nb = hamming_loss(y_test, y_pred_nb)
f1_macro_nb = f1_score(y_test, y_pred_nb, average='macro', zero_division=0)
f1_micro_nb = f1_score(y_test, y_pred_nb, average='micro', zero_division=0)
f1_weighted_nb = f1_score(y_test, y_pred_nb, average='weighted', zero_division=0)
jaccard_nb = jaccard_score(y_test, y_pred_nb, average='samples', zero_division=0)
subset_acc_nb = np.all(y_test == y_pred_nb, axis=1).mean()

In [ ]:

print(f"  Hamming Loss:        {hamming_nb:.4f}")
print(f"  F1-Score (Macro):    {f1_macro_nb:.4f}")
print(f"  F1-Score (Micro):    {f1_micro_nb:.4f}")
print(f"  F1-Score (Weighted): {f1_weighted_nb:.4f}")
print(f"  Jaccard Score:       {jaccard_nb:.4f}")
print(f"  Subset Accuracy:     {subset_acc_nb:.4f}")


  Hamming Loss:        0.0585
  F1-Score (Macro):    0.0908
  F1-Score (Micro):    0.2878
  F1-Score (Weighted): 0.2684
  Jaccard Score:       0.1855
  Subset Accuracy:     0.1595


## MODEL 2: RANDOM FOREST + TF-IDF


In [ ]:

rf_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
)

rf_model.fit(X_train_tfidf, y_train)


MultiOutputClassifier(estimator=RandomForestClassifier(max_depth=20, n_jobs=-1,
                                                       random_state=42))

In [ ]:
y_pred_rf = rf_model.predict(X_test_tfidf)


In [ ]:

# Evaluation
hamming_rf = hamming_loss(y_test, y_pred_rf)
f1_macro_rf = f1_score(y_test, y_pred_rf, average='macro', zero_division=0)
f1_micro_rf = f1_score(y_test, y_pred_rf, average='micro', zero_division=0)
f1_weighted_rf = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)
jaccard_rf = jaccard_score(y_test, y_pred_rf, average='samples', zero_division=0)
subset_acc_rf = np.all(y_test == y_pred_rf, axis=1).mean()


In [ ]:


print(f"  Hamming Loss:        {hamming_rf:.4f} ")
print(f"  F1-Score (Macro):    {f1_macro_rf:.4f}")
print(f"  F1-Score (Micro):    {f1_micro_rf:.4f}")
print(f"  F1-Score (Weighted): {f1_weighted_rf:.4f}")
print(f"  Jaccard Score:       {jaccard_rf:.4f}")
print(f"  Subset Accuracy:     {subset_acc_rf:.4f}")


  Hamming Loss:        0.0638 
  F1-Score (Macro):    0.0226
  F1-Score (Micro):    0.0750
  F1-Score (Weighted): 0.0720
  Jaccard Score:       0.0405
  Subset Accuracy:     0.0347


# APPROACH 2: PRETRAINED EMBEDDINGS + CLASSICAL ML

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

## EMBEDDINGS Model

In [ ]:
embed_model = SentenceTransformer('all-mpnet-base-v2', device=device)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Handle long texts - truncate for embedding model (max 756 tokens ≈ 1000 chars)
MAX_EMBED_LENGTH = 2000
X_train_truncated = [text[:MAX_EMBED_LENGTH] for text in X_train]
X_test_truncated = [text[:MAX_EMBED_LENGTH] for text in X_test]


In [ ]:
# Generate embeddings
X_train_emb = embed_model.encode(
    X_train_truncated,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/626 [00:00<?, ?it/s]

In [ ]:
X_test_emb = embed_model.encode(
    X_test_truncated,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

In [ ]:
X_train_emb.shape,X_test_emb.shape

((20015, 768), (5008, 768))

## MODEL 3: XGBOOST + EMBEDDINGS


In [ ]:
y_pred_xgb = np.zeros_like(y_test)

num_labels = y_train.shape[1]
print(f"Total labels to train: {num_labels}")

for i in range(num_labels):
    if i % 10 == 0:
        print(f"  Progress: {i}/{num_labels} classifiers trained...")

    xgb_clf = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        tree_method="hist",
        random_state=42,
        verbosity=0,
        device="cuda")

    xgb_clf.fit(X_train_emb, y_train[:, i])
    y_pred_xgb[:, i] = xgb_clf.predict(X_test_emb)


Total labels to train: 18
  Progress: 0/18 classifiers trained...
  Progress: 10/18 classifiers trained...


In [ ]:
y_pred_xgb[:, i] = xgb_clf.predict(X_test_emb)

In [ ]:
# Evaluation
hamming_xgb = hamming_loss(y_test, y_pred_xgb)
f1_macro_xgb = f1_score(y_test, y_pred_xgb, average='macro', zero_division=0)
f1_micro_xgb = f1_score(y_test, y_pred_xgb, average='micro', zero_division=0)
f1_weighted_xgb = f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
jaccard_xgb = jaccard_score(y_test, y_pred_xgb, average='samples', zero_division=0)
subset_acc_xgb = np.all(y_test == y_pred_xgb, axis=1).mean()


In [ ]:


print(f"  Hamming Loss:        {hamming_xgb:.4f} ")
print(f"  F1-Score (Macro):    {f1_macro_xgb:.4f}")
print(f"  F1-Score (Micro):    {f1_micro_xgb:.4f}")
print(f"  F1-Score (Weighted): {f1_weighted_xgb:.4f}")
print(f"  Jaccard Score:       {jaccard_xgb:.4f}")
print(f"  Subset Accuracy:     {subset_acc_xgb:.4f}")



  Hamming Loss:        0.0548 
  F1-Score (Macro):    0.2414
  F1-Score (Micro):    0.4480
  F1-Score (Weighted): 0.4281
  Jaccard Score:       0.3418
  Subset Accuracy:     0.2895


# APPROACH 3: TRANSFORMER FINE-TUNING


## Custom dataset class


In [ ]:
class MultiLabelDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)


##  Metrics function


In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).numpy()

    return {
        'f1_macro': f1_score(labels, predictions, average='macro', zero_division=0),
        'f1_micro': f1_score(labels, predictions, average='micro', zero_division=0),
        'hamming_loss': hamming_loss(labels, predictions),
        'jaccard': jaccard_score(labels, predictions, average='samples', zero_division=0)
    }


## Device

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

## DistilBERT model

In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=y_train.shape[1],
    problem_type="multi_label_classification",

).to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Handle text length - DistilBERT max is 512 tokens
# Truncate plots to ~2500 characters (roughly 500 tokens)
MAX_TRANSFORMER_LENGTH = 2500
X_train_transformer = [text[:MAX_TRANSFORMER_LENGTH] for text in X_train]
X_test_transformer = [text[:MAX_TRANSFORMER_LENGTH] for text in X_test]

In [ ]:
# Tokenize
train_encodings = tokenizer(
    X_train_transformer,
    truncation=True,
    padding=True,
    max_length=512,  # DistilBERT max
    return_tensors='pt'
)

test_encodings = tokenizer(
    X_test_transformer,
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors='pt'
)

In [ ]:
# Create datasets
train_dataset = MultiLabelDataset(train_encodings, y_train)
test_dataset = MultiLabelDataset(test_encodings, y_test)

In [ ]:

# Training arguments
training_args = TrainingArguments(
    output_dir='./results_distilbert',
    num_train_epochs=3,
    per_device_train_batch_size=16 if device == 'cuda' else 8,
    per_device_eval_batch_size=16 if device == 'cuda' else 8,
    warmup_steps=500,
    weight_decay=0.005,
    logging_dir='./logs_distilbert',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=torch.cuda.is_available(),  # Mixed precision on GPU
    report_to='none',  # Disable wandb
)


In [ ]:

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)


trainer.train()


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Hamming Loss,Jaccard,Runtime,Samples Per Second,Steps Per Second
1,0.128900,0.124481,0.193025,0.561230,0.049088,0.495873,17.659400,283.588000,17.724000
2,0.106600,0.118026,0.263642,0.597494,0.045616,0.534345,18.274500,274.043000,17.128000
3,0.078700,0.122886,0.313880,0.610507,0.046881,0.561635,17.666700,283.472000,17.717000


TrainOutput(global_step=3753, training_loss=0.12756764345603594, metrics={'train_runtime': 873.9935, 'train_samples_per_second': 68.702, 'train_steps_per_second': 4.294, 'total_flos': 7956274526484480.0, 'train_loss': 0.12756764345603594, 'epoch': 3.0})

In [ ]:

# Predict
predictions = trainer.predict(test_dataset)
y_pred_distilbert = (torch.sigmoid(torch.tensor(predictions.predictions)) > 0.5).numpy()


In [ ]:

# Evaluation
hamming_distilbert = hamming_loss(y_test, y_pred_distilbert)
f1_macro_distilbert = f1_score(y_test, y_pred_distilbert, average='macro', zero_division=0)
f1_micro_distilbert = f1_score(y_test, y_pred_distilbert, average='micro', zero_division=0)
f1_weighted_distilbert = f1_score(y_test, y_pred_distilbert, average='weighted', zero_division=0)
jaccard_distilbert = jaccard_score(y_test, y_pred_distilbert, average='samples', zero_division=0)
subset_acc_distilbert = np.all(y_test == y_pred_distilbert, axis=1).mean()


In [ ]:

print(f"  Hamming Loss:        {hamming_distilbert:.4f}")
print(f"  F1-Score (Macro):    {f1_macro_distilbert:.4f}")
print(f"  F1-Score (Micro):    {f1_micro_distilbert:.4f}")
print(f"  F1-Score (Weighted): {f1_weighted_distilbert:.4f}")
print(f"  Jaccard Score:       {jaccard_distilbert:.4f}")
print(f"  Subset Accuracy:     {subset_acc_distilbert:.4f}")



  Hamming Loss:        0.0469
  F1-Score (Macro):    0.3139
  F1-Score (Micro):    0.6105
  F1-Score (Weighted): 0.5865
  Jaccard Score:       0.5616
  Subset Accuracy:     0.4778


##   ROBERTA-BASE (768 embedding dimension)


In [ ]:

roberta_name = 'roberta-base'
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(
    roberta_name,
    num_labels=y_train.shape[1],
    problem_type="multi_label_classification"
).to(device)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:

# RoBERTa has same 512 token limit as BERT
MAX_ROBERTA_LENGTH = 2500
X_train_roberta = [text[:MAX_ROBERTA_LENGTH] for text in X_train]
X_test_roberta = [text[:MAX_ROBERTA_LENGTH] for text in X_test]


In [ ]:
# Tokenize
train_encodings_roberta = roberta_tokenizer(
    X_train_roberta,
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors='pt'
)

test_encodings_roberta = roberta_tokenizer(
    X_test_roberta,
    truncation=True,
    padding=True,
    max_length=512,
    return_tensors='pt'
)

In [ ]:

# Create datasets
train_dataset_roberta = MultiLabelDataset(train_encodings_roberta, y_train)
test_dataset_roberta = MultiLabelDataset(test_encodings_roberta, y_test)


In [ ]:






# Training arguments
training_args_roberta = TrainingArguments(
    output_dir='./results_roberta',
    num_train_epochs=3,
    per_device_train_batch_size=16 if device == 'cuda' else 8,
    per_device_eval_batch_size=16 if device == 'cuda' else 8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_roberta',
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=torch.cuda.is_available(),
    report_to='none',
)




In [ ]:
# Trainer
trainer_roberta = Trainer(
    model=roberta_model,
    args=training_args_roberta,
    train_dataset=train_dataset_roberta,
    eval_dataset=test_dataset_roberta,
    compute_metrics=compute_metrics,
)

# Train

trainer_roberta.train()

Epoch,Training Loss,Validation Loss,F1 Macro,F1 Micro,Hamming Loss,Jaccard,Runtime,Samples Per Second,Steps Per Second
1,0.134900,0.126776,0.190303,0.558088,0.049033,0.494708,35.094100,142.702000,8.919000
2,0.114500,0.118447,0.251796,0.595508,0.045949,0.532149,34.129000,146.737000,9.171000
3,0.093000,0.119038,0.283009,0.611425,0.046559,0.561402,34.134300,146.715000,9.170000


TrainOutput(global_step=3753, training_loss=0.13371563173757056, metrics={'train_runtime': 1956.4243, 'train_samples_per_second': 30.691, 'train_steps_per_second': 1.918, 'total_flos': 1.580077289327616e+16, 'train_loss': 0.13371563173757056, 'epoch': 3.0})

In [ ]:
# Predict
predictions_roberta = trainer_roberta.predict(test_dataset_roberta)
y_pred_roberta = (torch.sigmoid(torch.tensor(predictions_roberta.predictions)) > 0.5).numpy()


In [ ]:

# Evaluation
hamming_roberta = hamming_loss(y_test, y_pred_roberta)
f1_macro_roberta = f1_score(y_test, y_pred_roberta, average='macro', zero_division=0)
f1_micro_roberta = f1_score(y_test, y_pred_roberta, average='micro', zero_division=0)
f1_weighted_roberta = f1_score(y_test, y_pred_roberta, average='weighted', zero_division=0)
jaccard_roberta = jaccard_score(y_test, y_pred_roberta, average='samples', zero_division=0)
subset_acc_roberta = np.all(y_test == y_pred_roberta, axis=1).mean()


In [ ]:
print(f"  Hamming Loss:        {hamming_roberta:.4f} ")
print(f"  F1-Score (Macro):    {f1_macro_roberta:.4f}")
print(f"  F1-Score (Micro):    {f1_micro_roberta:.4f}")
print(f"  F1-Score (Weighted): {f1_weighted_roberta:.4f}")
print(f"  Jaccard Score:       {jaccard_roberta:.4f}")
print(f"  Subset Accuracy:     {subset_acc_roberta:.4f}")

  Hamming Loss:        0.0466 
  F1-Score (Macro):    0.2830
  F1-Score (Micro):    0.6114
  F1-Score (Weighted): 0.5856
  Jaccard Score:       0.5614
  Subset Accuracy:     0.4772


# FINAL COMPARISON


In [ ]:

# Create comparison table
results_dict = {
    'MultinomialNB + TF-IDF': {
        'Hamming Loss': hamming_nb,
        'F1-Macro': f1_macro_nb,
        'F1-Micro': f1_micro_nb,
        'F1-Weighted': f1_weighted_nb,
        'Jaccard': jaccard_nb,
        'Subset Acc': subset_acc_nb
    },
    'Random Forest + TF-IDF': {
        'Hamming Loss': hamming_rf,
        'F1-Macro': f1_macro_rf,
        'F1-Micro': f1_micro_rf,
        'F1-Weighted': f1_weighted_rf,
        'Jaccard': jaccard_rf,
        'Subset Acc': subset_acc_rf
    },
    'XGBoost + Embeddings': {
        'Hamming Loss': hamming_xgb,
        'F1-Macro': f1_macro_xgb,
        'F1-Micro': f1_micro_xgb,
        'F1-Weighted': f1_weighted_xgb,
        'Jaccard': jaccard_xgb,
        'Subset Acc': subset_acc_xgb
    },
    'DistilBERT Transformer': {
        'Hamming Loss': hamming_distilbert,
        'F1-Macro': f1_macro_distilbert,
        'F1-Micro': f1_micro_distilbert,
        'F1-Weighted': f1_weighted_distilbert,
        'Jaccard': jaccard_distilbert,
        'Subset Acc': subset_acc_distilbert
    },

    'RoBERTa Transformer': {
        'Hamming Loss': hamming_roberta,
        'F1-Macro': f1_macro_roberta,
        'F1-Micro': f1_micro_roberta,
        'F1-Weighted': f1_weighted_roberta,
        'Jaccard': jaccard_roberta,
        'Subset Acc': subset_acc_roberta
    }}

comparison_df = pd.DataFrame(results_dict).T
comparison_df = comparison_df.sort_values('F1-Macro', ascending=False)
comparison_df

,Hamming Loss,F1-Macro,F1-Micro,F1-Weighted,Jaccard,Subset Acc
DistilBERT Transformer,0.046881,0.313880,0.610507,0.586451,0.561635,0.477835
RoBERTa Transformer,0.046559,0.283009,0.611425,0.585590,0.561402,0.477236
XGBoost + Embeddings,0.054846,0.241352,0.447968,0.428113,0.341820,0.289537
MultinomialNB + TF-IDF,0.058517,0.090805,0.287836,0.268361,0.185503,0.159545
Random Forest + TF-IDF,0.063798,0.022625,0.074956,0.071999,0.040535,0.034744


In [ ]:
# Find best model
best_model = comparison_df['F1-Macro'].idxmax()
best_model

'DistilBERT Transformer'

# SAVE MODEL AND ARTIFACTS FOR DEPLOYMENT


In [ ]:

model.save_pretrained('distilbert_genre_classifier')
tokenizer.save_pretrained('distilbert_genre_classifier')

# Save MultiLabelBinarizer
joblib.dump(mlb, 'mlb_encoder.pkl')

# Save genre mapping for easy access
genre_info = {
    'classes': mlb.classes_.tolist(),
    'num_classes': len(mlb.classes_),
    'max_sequence_length': 2000,
    'model_name': 'distilbert-base-uncased'
}

with open('genre_info.json', 'w') as f:
    json.dump(genre_info, f, indent=2)
